In [159]:
from course_utils.paths import get_project_root, get_data_dir

print("프로젝트 루트 :", get_project_root())
print("데이터 폴더 :", get_data_dir())

프로젝트 루트 : C:\dev\llm-data-analysis-course
데이터 폴더 : C:\dev\llm-data-analysis-course\data


In [160]:
RAW_DIR =  get_project_root() / "data" / "raw"
PROCESSED_DIR =  get_project_root() / "data" / "processed"
REPORT_DIR =  get_project_root() / "reports"

print(RAW_DIR)
print(PROCESSED_DIR)
print(REPORT_DIR)

C:\dev\llm-data-analysis-course\data\raw
C:\dev\llm-data-analysis-course\data\processed
C:\dev\llm-data-analysis-course\reports


In [161]:
import pandas as pd

customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")

In [162]:
print(customers.info())
print(products.info())
print(orders.info())
print(order_items.info())

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 7.2 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 3.3 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------

In [163]:
print(customers.shape)
print(products.shape)
print(orders.shape)
print(order_items.shape)

(150, 6)
(100, 4)
(300, 5)
(764, 5)


In [164]:
raw_data = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

for name, df in raw_data.items():
    print(name)
    print("shape:", df.shape)
    print("missing:", int(df.isna().sum().sum()))
    print("duplicated rows:", int(df.duplicated().sum()))
    print()

customers
shape: (150, 6)
missing: 0
duplicated rows: 0

products
shape: (100, 4)
missing: 0
duplicated rows: 0

orders
shape: (300, 5)
missing: 0
duplicated rows: 0

order_items
shape: (764, 5)
missing: 0
duplicated rows: 0



### STEP 4. 원본을 복사해 전처리용 DataFrame 만들기

In [165]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

### STEP 5. 문자열 공백과 빈 문자열 점검

In [166]:
customers_clean["name"].head()

0    김수민
1    김정호
2    이경수
3    조영호
4    이예원
Name: name, dtype: str

In [ ]:
customers_clean["name"] = customers_clean["name"].where(
    customers_clean["name"].isna(),
    customers_clean["name"].astype(str).str.strip(),
)


In [168]:
import numpy as np

# 위 코드에 대한 예시 코드

# 새로운 데이터 프레임 만들기 -> 리스트 + 딕셔너리
df = pd.DataFrame({
    'name' : ['    홍길동    ', '    이영희    ', np.nan, '    박철수    ']
})

print(df.info())

print("\n--- [처리전]--- ")
print(df["name"].to_list())

<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    3 non-null      str  
dtypes: str(1)
memory usage: 164.0 bytes
None

--- [처리전]--- 
['    홍길동    ', '    이영희    ', nan, '    박철수    ']


In [169]:
# 기존 공백 제거 코드 실행

df["name"] = df["name"].where(
    df["name"].isna(),                      # null인지 아닌지
    df["name"].astype(str).str.strip(),     # 공백제거 / null이면 공백제거 할 필요없음 null일 경우 이 줄은 실행 안 함
)


print("\n--- [처리후]--- ")
print(df["name"].to_list())


--- [처리후]--- 
['홍길동', '이영희', nan, '박철수']


In [173]:
# 검증코드
print(df["name"].str.startswith(" ") | df["name"].str.endswith(" "))    # .str 사용하면 string과 연관된 메소드 사용 가능

print((df["name"].str.startswith(" ") | df["name"].str.endswith(" ")).sum())

print((df["name"].str.startswith(" ").sum()) + (df["name"].str.endswith(" ").sum()))

0    False
1    False
2    False
3    False
Name: name, dtype: bool
0
0


In [181]:
cols = customers_clean.columns.to_list();
print(cols)

for col in cols:
    print(col)

['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
customer_id
name
gender
age
city
signup_date


In [179]:
cols = customers_clean.columns.to_list();
for col in cols:
    if customers_clean[col].dtype == "str":
        print(customers_clean[col].dtype)

str
str
str
str


In [171]:
customers_clean["name"] = customers_clean["name"].replace(
    "",
    pd.NA,
)

customers_clean.isna().sum()

customer_id    0
name           0
gender         0
age            0
city           0
signup_date    0
dtype: int64

In [172]:
customers_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 7.2 KB


In [185]:
print("" == " ".strip())

# "" -> 빈문자열   |  " " -> 공란   이 둘은 같지 않다.
# " ".strip() 활용하여 공란 없애면 빈문자열과 같음

True


In [186]:
df.isna().sum()

name    1
dtype: int64